In [3]:
import pandas as pd
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import joblib

# Loading the specific data sets provided in the sources [1, 2]
df_fake = pd.read_csv("fake.csv") 
df_true = pd.read_csv("true.csv")

# Assigning numerical classes: 0 for fake, 1 for true [3]
df_fake['class'] = 0
df_true['class'] = 1

# Combining data sets [3]
df_mrg = pd.concat([df_fake, df_true], axis=0)

# Dropping unnecessary columns based on the new file structure (id, title, subject, and the original label) [1-3]
# We keep only 'text' and the newly created 'class' [3]
df = df_mrg.drop(['id', 'title', 'subject', 'label'], axis=1)
df = df.reset_index(drop=True) 

# Text cleaning function as described in the video workflow [4, 5]
def clean_text(text):
    text = text.lower() # Uniformity [4]
    text = re.sub('\[.*?\]', '', text) # Remove text in square brackets [4]
    text = re.sub('\\W', ' ', text) # Remove non-word characters [4]
    text = re.sub('https?://\S+|www\.\S+', '', text) # Remove URLs [4]
    text = re.sub('<.*?>+', '', text) # Remove HTML tags [4]
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text) # Remove punctuation [4]
    text = re.sub('\n', '', text) # Remove new lines [4]
    text = re.sub('\w*\d\w*', '', text) # Remove words containing digits [5]
    return text

df['text'] = df['text'].apply(clean_text) 

# Defining variables [5]
X = df['text']
y = df['class']

# Splitting data (test size 0.25, random state 42) [5]
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Vectorization using TF-IDF [5]
vectorization = TfidfVectorizer()
xv_train = vectorization.fit_transform(x_train)
xv_test = vectorization.transform(x_test)

# Training with Logistic Regression [5]
LR = LogisticRegression()
LR.fit(xv_train, y_train)

# Evaluation [6]
pred_lr = LR.predict(xv_test)
print(classification_report(y_test, pred_lr))

# Saving the model and vectorizer for deployment [6]
joblib.dump(LR, 'LR_model.jb')
joblib.dump(vectorization, 'vectorizer.jb')



              precision    recall  f1-score   support

           0       1.00      1.00      1.00        27
           1       1.00      1.00      1.00        23

    accuracy                           1.00        50
   macro avg       1.00      1.00      1.00        50
weighted avg       1.00      1.00      1.00        50



['vectorizer.jb']